# Diabetes Patient Data Analysis

Exploratory data analysis on the Pima Indians Diabetes Database from the National Institute of Diabetes and Digestive and Kidney Diseases. The dataset contains medical predictor variables and one target variable (Outcome). Analysis includes data cleaning, missing value treatment, outlier detection, correlation analysis, and feature engineering.

This dataset is originally from the **National Institute of Diabetes and Digestive and Kidney Diseases.** The dataset consist of several medical predictor **(independent)** variables and one target **(dependent)** variable, Outcome. 

Independent variables include:

- Number of times pregnant.
- Plasma glucose concentration a 2 hours in an oral glucose tolerance test.
- Diastolic blood pressure (mm Hg).
- Triceps skinfold thickness (mm).
- Two-Hour serum insulin (mu U/ml).
- Body mass index (weight in kg/(height in m)^2).
- Diabetes pedigree function.
- Age (years).
- Outcome: Class variable (0 or 1).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
df = pd.read_csv("Dataset/diabetes.csv")

In [ ]:
# Let's look at the first few rows
df.head()

### 1. Identify the total number of records? 

In [ ]:
df.shape[0]

### 2. Display data types for all columns? 

In [ ]:
df.dtypes

### 3.  Check the dataset for the missing values? 

In [ ]:
df.isnull().values.any()

### 4. Display descriptive statistics include those that summarize the central tendency, dispersion and shape of a dataset such as total count, min, max, standard deviation, max and quartiles?

In [ ]:
df.describe()

As you probably have noticed, many columns have a **minimum value of 0** which is clearly not logical. Those are essentially **missing values** in our dataset. 



### 5. Identify columns with missing values and find out their frequency? 

In [ ]:
#listing columns with missing values(0)
missing_col = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

In [ ]:
def show_zero(col_name):
    print('%s: %d'%(col_name, df[df[col_name]==0].shape[0])) #Identifying the number of '0' values

In [ ]:
#loopig through each name in missing_col and calling the function
for col_name in missing_col:
    show_zero(col_name)

### 6. Identify the number of positive and negative patients based on target (dependent) variable, outcome? Plot the frequencies using a bar chart?

In [ ]:
#number of patients in each outcome
outcome= df['Outcome'].value_counts()
outcome

In [ ]:
#plotting the above results
outcome.plot.bar(color="green", xlabel= 'Patient Outcome', ylabel='Frequency')

### 7. Display correlation among all dependent and independent variables using the correlation matrix? Illustrate correlation data using a heatmap plot?  

In [ ]:
correlation = df.corr()
correlation

In [ ]:
sns.heatmap(correlation, cmap="Greens", annot=True)

We have observed earlier that missing values in this dataset are represented by 0. This is clearly not a good practice and negatively impacts our data analysis. 



### 8. Replace these zero values with NAN?

In [ ]:
df[missing_col] = df[missing_col].replace(0, np.NaN)

### 9. Check out the variables for 0 value once again?  

In [ ]:
for col_name in missing_col:
    show_zero(col_name)

### 10. Check out the missing values counts per variable?

In [ ]:
df.isna().sum()

### 11. Identify the mean value for each variable? 

In [ ]:
df.describe().T["mean"]

### 12. For the following columns, fill missing values with mean column values?  
- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI

In [ ]:
df_count_mean = df[missing_col].describe().T[["count", "mean"]]
df_count_mean["count"] = df_count_mean["count"].apply(lambda value: int(df.shape[0]-value))
df_count_mean.rename(columns={"count": "missing"}, inplace=True)
df_count_mean

### 13. Display descriptive statistics and checkout the min values of the aforementioned columns once again? Any ZEROS ? 

In [ ]:
df[missing_col].describe()

In [ ]:
#checking for 0 in any of the mentioned columns
df[missing_col].describe().loc["min"] == 0

### 14. Check out the missing values counts per variable, once again? 

In [ ]:
df.isna().sum()

### 15. Plot data distribution of each variable? Explain your thoughts 

In [ ]:
for col in df.columns:
    plot = sns.displot(df, x=col, color="green")
    plt.show()
    
'''comparing the graph plots we see that:
> The data provided pertains more to the younger population aged between 20-40
> More tha half of the population is Diabetic patients'''

In [ ]:
A Boxplot is a method for graphically depicting groups of numerical data through their quartiles. 

### 16. Plot the Boxplot for each variable? interpret the diagrams 

In [ ]:
for column in df:
    plt.figure(figsize=(17,1))
    sns.boxplot(data=df, x=column, color="green")

### 17. Plot demographic and distribution of diabetics/nondiabetics across age variable?

In [ ]:
plot = sns.displot(df, x="Age", hue="Outcome")
plt.show()

### 18. Investigate how the number of pregnancies impacts diabetes? Demonstrate the relationship with a plot. 

In [ ]:
plot = sns.kdeplot(df, x="Pregnancies", hue="Outcome")
plt.xticks(np.arange(0,20,1))
plt.show()

In [ ]:
plot = sns.stripplot(df, x="Pregnancies", hue="Outcome")
plt.xticks(np.arange(0,20,1))
plt.show()

### 19. Check if there are any outliers in our dataset. Any data point outside 25% and 75% quarters can be considered an outlier. Remove the outliers from our dataset. 

In [ ]:
df.describe()

In [ ]:
df75 = df["DiabetesPedigreeFunction"] <= df["DiabetesPedigreeFunction"].quantile(0.75)
df25 = df["DiabetesPedigreeFunction"] >= df["DiabetesPedigreeFunction"].quantile(0.25)

df[df25 & df75]

### 20. Create a new categorical variable based on BMI using the following criteria. Name the new variable “BMI_tier” and add it to our dataset as a new column. 

- BMI = 0  then “NA”
- 0 < BMI < 18.5 then “Under Weight”
- 18.5 <= BMI < 25 then “Normal”
- 25 <= BMI < 30 then “Overweight” 
- 30 <= BMI then “Obese”


In [ ]:
#creating a function to define the required criteria
def calc_tier(BMI):
    if np.isnan(BMI) or BMI < 0:
        return "NA"
    elif 0 < BMI < 18.5:
        return "Under Weight"
    elif 18.5 <= BMI < 25:
        return "Normal"
    elif 25 <= BMI < 30:
        return "Overweight"
    else:
        return "Obese"

In [ ]:
#defining new column to the dataset
df["BMI_tier"] = df["BMI"].apply(calc_tier)

In [ ]:
df.head()

### 21. Plot the BMI_tier histogram ?  

In [ ]:
sns.histplot(df, x="BMI_tier", color="green")

### 22. How many obese individuals exist in our dataset?  Use  Piechart to illustrate the proportion of each BMI tier? 

In [ ]:
df["BMI_tier"].value_counts()["Obese"]

In [ ]:
df["BMI_tier"].value_counts().plot(kind='pie',colors=sns.color_palette('pastel'))

### 23. Create a new categorical variable based on Oral Glucose Tolerance Test (Glucose) using the following criteria, suggested by DIABETES UK. Name the new variable “OGTT_tier” and add it to our dataset as a new column. 

- Glucose == 0 then “NA”
- Glucose < 140 then “Normal”
- 140 <= Glucose < 198 then “Impaired Glucose Tolerance”
- 198 <= Glucose then “Diabetic Level”


In [ ]:
#creating a function to define the required criteria
def cal_ogtt(Glucose):
    if np.isnan(Glucose) or Glucose < 0:
        return "NA"
    elif Glucose < 140:
        return "Normal"
    elif 140 <= Glucose < 198:
        return "Impaired Glucose Tolerance"
    else:
        return "Diabetic Level"

# new column
df["OGTT_tier"] = df["Glucose"].apply(cal_ogtt)

In [ ]:
df.head()

### 24.How many individuals are categorized as “Diabetic Level” in our newly created variable, “OGTT_tier”?  Plot the “OGTT_tier” histogram?   

In [ ]:
df[df["OGTT_tier"] == "Diabetic Level"]

In [ ]:
ax = sns.displot(df, x="OGTT_tier", color="green")
plt.xticks(rotation=90)
plt.show()

### 25. Out of those who categorized as “Impaired Glucose Tolerance”, how many of them are actually diabetes? What about those with “Normal” OGTT_tier?  

In [ ]:
impaired = df["OGTT_tier"] == "Impaired Glucose Tolerance"
df[impaired]["Outcome"].value_counts().loc[1]